# Normalization from Scratch

Stack 32 transformer layers and the residual stream's variance grows additively with depth — softmaxes saturate, gradients explode, training diverges. Normalization layers tame that drift. This notebook implements every major scheme covered in the [normalization blog post](https://github.com/david-hoangt/david-hoangt.github.io) and runs the small experiments behind the post's claims:

$$\text{Each scheme picks an axis to pool stats over, then re-scales (and sometimes re-centers).}$$

| Section | Scheme | Axis pooled | Parameters per layer |
|---------|--------|-------------|----------------------|
| §1 | LayerNorm | feature dim, per token | `2 · d_model` (γ, β) |
| §2 | RMSNorm | feature dim, per token | `d_model` (γ only) |
| §3 | Pre-Norm vs Post-Norm | placement, not axis | same as norm used |
| §4 | Peri-LN | adds a second norm post-sublayer | `2 ·` norm params per block |
| §5 | QK-Norm (cosine + LayerNorm variants) | head dim of Q, K only | `n_heads` (g) or `2 · d_k` |
| §6 | DyT (Dynamic Tanh) | none — element-wise tanh | `2 · d_model + 1` (γ, β, α) |

Running example: *"The CEO announced record earnings on Friday"* — 7 tokens, `d_model = 64`.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

torch.manual_seed(42)

B, T, d_model = 2, 7, 64
tokens = ["The", "CEO", "announced", "record", "earnings", "on", "Friday"]


## §1 — LayerNorm

For each token, pool statistics across the feature dim — no batch coupling, no padding pollution.

$$y = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta, \quad \mu = \tfrac{1}{d}\sum_k x_k, \quad \sigma^2 = \tfrac{1}{d}\sum_k (x_k - \mu)^2$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| x | $(B,\, T,\, d_{\text{model}})$ | Residual stream |
| μ, σ² | $(B,\, T,\, 1)$ | Per-token statistics over the last dim |
| γ, β | $(d_{\text{model}},)$ | Learnable scale and shift |

> Two passes over the feature dim, two parameter vectors per layer.


In [ ]:
class LayerNorm(nn.Module):
    """
    Layer Normalization (Ba, Kiros, Hinton, 2016).

    y = γ ⊙ (x - μ) / sqrt(σ² + ε) + β
    where μ, σ² are computed per token over the last dim.

    Args:
        d_model: Feature dimension.
        eps:     Numerical stability constant.
    """

    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # [d_model]
        self.beta  = nn.Parameter(torch.zeros(d_model))  # [d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mu  = x.mean(dim=-1, keepdim=True)                  # [B, T, 1]
        var = x.var(dim=-1, keepdim=True, unbiased=False)   # [B, T, 1]
        x_hat = (x - mu) / torch.sqrt(var + self.eps)        # [B, T, d_model]
        return self.gamma * x_hat + self.beta


In [ ]:
# ── shape check + per-token statistics ──────────────────────────────────────
x = torch.randn(B, T, d_model) * 5 + 3   # arbitrary mean and scale
ln = LayerNorm(d_model=d_model)
y = ln(x)

print(f"Input  shape: {x.shape}")              # [2, 7, 64]
print(f"Output shape: {y.shape}")              # [2, 7, 64]

print(f"\nInput  per-token mean  (max abs): {x.mean(dim=-1).abs().max().item():.3f}")
print(f"Input  per-token std   (mean):    {x.std(dim=-1).mean().item():.3f}")
print(f"Output per-token mean  (max abs): {y.mean(dim=-1).abs().max().item():.2e}")
print(f"Output per-token std   (mean):    {y.std(dim=-1).mean().item():.3f}")

assert y.shape == x.shape
assert y.mean(dim=-1).abs().max().item() < 1e-5   # re-centered
assert abs(y.std(dim=-1).mean().item() - 1.0) < 1e-2  # re-scaled


## §2 — RMSNorm

Drop the mean computation entirely. One pass over features, one parameter vector.

$$y = \gamma \odot \frac{x}{\text{RMS}(x)}, \quad \text{RMS}(x) = \sqrt{\tfrac{1}{d}\sum_k x_k^2 + \varepsilon}$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| x | $(B,\, T,\, d_{\text{model}})$ | Residual stream |
| RMS(x) | $(B,\, T,\, 1)$ | Per-token root-mean-square |
| γ | $(d_{\text{model}},)$ | Learnable scale (no β) |

> Re-scaling invariance is preserved; re-centering invariance is dropped.


In [ ]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (Zhang & Sennrich, NeurIPS 2019).

    y = γ ⊙ x / sqrt(mean(x²) + ε)

    Args:
        d_model: Feature dimension.
        eps:     Numerical stability constant.
    """

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # [d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)  # [B, T, 1]
        return self.gamma * (x / rms)


In [ ]:
# ── compare LayerNorm vs RMSNorm on the same input ──────────────────────────
x = torch.randn(B, T, d_model) * 5 + 3
ln  = LayerNorm(d_model=d_model)
rms = RMSNorm(d_model=d_model)

y_ln  = ln(x)
y_rms = rms(x)

print(f"{'metric':<28} {'LayerNorm':>12} {'RMSNorm':>12}")
print("-" * 56)
print(f"{'per-token mean (max abs)':<28} {y_ln.mean(dim=-1).abs().max():>12.2e} {y_rms.mean(dim=-1).abs().max():>12.2e}")
print(f"{'per-token RMS (mean)':<28} {y_ln.pow(2).mean(dim=-1).sqrt().mean():>12.3f} {y_rms.pow(2).mean(dim=-1).sqrt().mean():>12.3f}")
print(f"{'parameter count':<28} {sum(p.numel() for p in ln.parameters()):>12d} {sum(p.numel() for p in rms.parameters()):>12d}")

# RMSNorm preserves the original mean (does NOT re-center)
mean_in   = x.mean(dim=-1)
mean_rms  = y_rms.mean(dim=-1)
mean_ln   = y_ln.mean(dim=-1)
print(f"\ninput mean (sample): {mean_in[0, :3].tolist()}")
print(f"RMSNorm mean still nonzero (sample): {mean_rms[0, :3].tolist()}")
print(f"LayerNorm mean ≈ 0 (sample):        {mean_ln[0, :3].tolist()}")


## §3 — Pre-Norm vs Post-Norm: the residual variance experiment

The placement of the norm changes how variance accumulates through depth. Run the same random sublayer through 32 layers under three regimes and watch what happens to the residual stream's RMS:

$$\text{Post-Norm:} \quad x \leftarrow \text{Norm}(x + \text{Sub}(x)) \qquad \text{Pre-Norm:} \quad x \leftarrow x + \text{Sub}(\text{Norm}(x))$$

| Regime | Stability at init | Modern usage |
|--------|--------------------|--------------|
| No norm | Diverges | None |
| Post-Norm | Needs LR warmup | Original Transformer (2017) |
| Pre-Norm | Stable, no warmup | LLaMA, Mistral, Qwen, SmolLM3 |

> Pre-Norm protects the gradient path; the trade-off is unbounded residual growth at extreme depth (motivates Peri-LN in §4).


In [ ]:
def init_linear_(layer: nn.Linear, gain: float) -> None:
    """Kaiming-normal init with a controllable gain — `gain > 1` mimics late-training drift."""
    nn.init.kaiming_normal_(layer.weight, nonlinearity="linear")
    with torch.no_grad():
        layer.weight.mul_(gain)


class Block(nn.Module):
    """
    Single transformer block with configurable norm placement.

    placement:
      - "pre"  → x + Sublayer(Norm(x))      (LLaMA-style)
      - "post" → Norm(x + Sublayer(x))      (Vaswani 2017)
      - "none" → x + Sublayer(x)            (no normalization)

    `gain` scales the sublayer's linear weights — gain=1 is a clean init,
    larger gain mimics what training does to weights over many steps.
    """

    def __init__(self, d_model: int, placement: str = "pre", gain: float = 2.0):
        super().__init__()
        assert placement in ("pre", "post", "none")
        self.placement = placement
        self.norm = RMSNorm(d_model)
        lin1 = nn.Linear(d_model, 4 * d_model, bias=False)
        lin2 = nn.Linear(4 * d_model, d_model, bias=False)
        init_linear_(lin1, gain)
        init_linear_(lin2, gain)
        self.sublayer = nn.Sequential(lin1, nn.GELU(), lin2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.placement == "pre":
            return x + self.sublayer(self.norm(x))
        if self.placement == "post":
            return self.norm(x + self.sublayer(x))
        return x + self.sublayer(x)


def stack_rms_per_layer(placement: str, n_layers: int = 32, gain: float = 2.0, seed: int = 0) -> list[float]:
    """Run a stack of `n_layers` blocks at init and record per-token RMS at each depth."""
    torch.manual_seed(seed)
    blocks = nn.ModuleList([Block(d_model, placement=placement, gain=gain) for _ in range(n_layers)])
    x = torch.randn(B, T, d_model)
    rms_history = [x.pow(2).mean(dim=-1).sqrt().mean().item()]
    with torch.no_grad():
        for blk in blocks:
            x = blk(x)
            rms_history.append(x.pow(2).mean(dim=-1).sqrt().mean().item())
    return rms_history


In [ ]:
# ── variance growth: no-norm vs Post-Norm vs Pre-Norm across 32 layers ──────
# gain=2 inflates the sublayer's linear weights to mimic mid-training drift —
# clean random init produces sublayer outputs too small to expose the failure mode.
n_layers = 32
gain = 2.0
hist_none = stack_rms_per_layer("none", n_layers=n_layers, gain=gain)
hist_post = stack_rms_per_layer("post", n_layers=n_layers, gain=gain)
hist_pre  = stack_rms_per_layer("pre",  n_layers=n_layers, gain=gain)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hist_none, label="no normalization", color="#EF4444", linewidth=2)
ax.plot(hist_post, label="Post-Norm (RMSNorm)", color="#1E3A8A", linewidth=2)
ax.plot(hist_pre,  label="Pre-Norm (RMSNorm)",  color="#10B981", linewidth=2)
ax.set_yscale("log")
ax.set_xlabel("Layer depth")
ax.set_ylabel("Residual stream RMS  (log scale)")
ax.set_title(f"Residual stream magnitude across 32 layers (sublayer gain={gain})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final residual RMS at layer {n_layers}:")
print(f"  no-norm:   {hist_none[-1]:.3e}   ← grows multiplicatively, blows up")
print(f"  Post-Norm: {hist_post[-1]:.3e}   ← every block renormalizes to ~1")
print(f"  Pre-Norm:  {hist_pre[-1]:.3e}   ← grows additively, but bounded per step")


## §4 — Peri-LN block (Gemma-style)

Pre-Norm protects gradients; Post-Norm clamps the residual. Peri-LN keeps both — a norm before *and* after each sublayer. The extra post-norm is what stops the Pre-Norm residual from drifting at depth.

$$x \leftarrow x + \text{Norm}_{\text{post}}\bigl(\text{Sub}\bigl(\text{Norm}_{\text{pre}}(x)\bigr)\bigr)$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| Norm_pre, Norm_post | $(d_{\text{model}},)$ params | Two RMSNorms per sublayer |

> Twice the norm parameters per block, negligible compute, much tighter residual variance at scale.


In [ ]:
class PeriLNBlock(nn.Module):
    """
    Gemma-style block: RMSNorm both before AND after each sublayer.

      x = x + RMSNorm_post(Sublayer(RMSNorm_pre(x)))
    """

    def __init__(self, d_model: int, gain: float = 2.0):
        super().__init__()
        self.pre_norm  = RMSNorm(d_model)
        self.post_norm = RMSNorm(d_model)
        lin1 = nn.Linear(d_model, 4 * d_model, bias=False)
        lin2 = nn.Linear(4 * d_model, d_model, bias=False)
        init_linear_(lin1, gain)
        init_linear_(lin2, gain)
        self.sublayer = nn.Sequential(lin1, nn.GELU(), lin2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.post_norm(self.sublayer(self.pre_norm(x)))


def stack_peri_per_layer(n_layers: int = 32, gain: float = 2.0, seed: int = 0) -> list[float]:
    torch.manual_seed(seed)
    blocks = nn.ModuleList([PeriLNBlock(d_model, gain=gain) for _ in range(n_layers)])
    x = torch.randn(B, T, d_model)
    rms_history = [x.pow(2).mean(dim=-1).sqrt().mean().item()]
    with torch.no_grad():
        for blk in blocks:
            x = blk(x)
            rms_history.append(x.pow(2).mean(dim=-1).sqrt().mean().item())
    return rms_history


In [ ]:
# ── Pre-Norm vs Peri-LN: residual control under sublayer drift ──────────────
# Sweep `gain` from 1 (clean init) to 4 (heavy training drift). Pre-Norm's residual
# scales with the sublayer magnitude; Peri-LN's post-norm clamps each step to ~RMS=1
# so the depth-32 residual is bounded by ~sqrt(32) regardless of gain.
gains = [1.0, 2.0, 3.0, 4.0]
fig, ax = plt.subplots(figsize=(11, 5.5))
colors = plt.cm.RdYlBu_r(np.linspace(0.15, 0.85, len(gains)))

for c, g in zip(colors, gains):
    hp_pre  = stack_rms_per_layer("pre",  n_layers=n_layers, gain=g)
    hp_peri = stack_peri_per_layer(n_layers=n_layers, gain=g)
    ax.plot(hp_pre,  "--", color=c, label=f"Pre-Norm   (gain={g})", linewidth=1.6)
    ax.plot(hp_peri, "-",  color=c, label=f"Peri-LN    (gain={g})", linewidth=2.0)

ax.set_xlabel("Layer depth")
ax.set_ylabel("Residual stream RMS")
ax.set_title("Pre-Norm scales with sublayer drift; Peri-LN's post-norm caps it")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Numerical summary
print(f"{'gain':>6} {'Pre-Norm':>14} {'Peri-LN':>14}")
print("-" * 36)
for g in gains:
    hp_pre  = stack_rms_per_layer("pre",  n_layers=n_layers, gain=g)
    hp_peri = stack_peri_per_layer(n_layers=n_layers, gain=g)
    print(f"{g:>6.1f} {hp_pre[-1]:>14.3f} {hp_peri[-1]:>14.3f}")


## §5 — QK-Norm: bound the attention logits

Two variants, both normalize `Q` and `K` along the head dim before the dot product:

**Variant A — Cosine (Henry et al., 2020)**
$$\text{scores} = g \cdot \bigl(\hat{Q} \, \hat{K}^\top\bigr), \quad \hat{Q} = Q / \lVert Q \rVert,\; \hat{K} = K / \lVert K \rVert \in [-1, 1]$$

**Variant B — LayerNorm/RMSNorm on Q, K (ViT-22B, Gemma)**
$$\text{scores} = \bigl(\text{LN}(Q) \, \text{LN}(K)^\top\bigr) / \sqrt{d_k}$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| Q, K, V | $(B,\, h,\, T,\, d_k)$ | Per-head projections |
| g | $(h, 1, 1)$ | Learnable scalar per head (Variant A) |
| scores | $(B,\, h,\, T,\, T)$ | Attention logits |

> Variant A bounds logits to $[-g, g]$ by construction; Variant B keeps the standard scaling but caps Q/K magnitudes.


In [ ]:
class QKNormCosine(nn.Module):
    """
    Cosine QK-Norm (Variant A — Henry et al., EMNLP Findings 2020).

    Q_hat = F.normalize(Q, p=2, dim=-1)   # unit vectors along head dim
    K_hat = F.normalize(K, p=2, dim=-1)
    scores = g * (Q_hat @ K_hat.T)        # bounded to [-g, g]

    Args:
        d_model:    Embedding dimension.
        n_heads:    Number of heads.
        init_scale: Initial value of g (typical 10–20 for sharp attention).
    """

    def __init__(self, d_model: int, n_heads: int, init_scale: float = 10.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.g = nn.Parameter(torch.full((n_heads, 1, 1), init_scale))  # [h, 1, 1]

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # [B, h, T, d_k]
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        Q_hat = F.normalize(Q, p=2, dim=-1)
        K_hat = F.normalize(K, p=2, dim=-1)

        scores = self.g * torch.matmul(Q_hat, K_hat.transpose(-2, -1))  # [B, h, T, T]
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = torch.matmul(weights, V).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(context), scores


In [ ]:
class QKNormLayerNorm(nn.Module):
    """
    LayerNorm QK-Norm (Variant B — ViT-22B, Gemma 2/3).

    Q_hat = LayerNorm(Q, dim=d_k)         # not unit-length, just controlled magnitude
    K_hat = LayerNorm(K, dim=d_k)
    scores = (Q_hat @ K_hat.T) / sqrt(d_k)

    Args:
        d_model: Embedding dimension.
        n_heads: Number of heads.
    """

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = LayerNorm(self.d_k)   # applies along last dim of [B, h, T, d_k]
        self.k_norm = LayerNorm(self.d_k)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # [B, h, T, d_k]
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        Q = self.q_norm(Q)
        K = self.k_norm(K)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # [B, h, T, T]
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = torch.matmul(weights, V).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(context), scores


In [ ]:
# ── logit-magnitude experiment: standard MHA vs QK-Norm A vs QK-Norm B ──────
# Simulate the failure mode: as Q/K weights drift to large magnitudes during training,
# unnormalized scores grow without bound. We mimic late-training weights by scaling them up.

class StandardMHA(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads; self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        return scores


def max_logit_at_weight_scale(attn_cls, scale: float, **kwargs) -> float:
    torch.manual_seed(0)
    attn = attn_cls(d_model=64, n_heads=4, **kwargs)
    # Inflate Q/K projection weights to mimic late-training drift
    with torch.no_grad():
        attn.W_q.weight.mul_(scale)
        attn.W_k.weight.mul_(scale)
    x = torch.randn(B, T, d_model)
    out = attn(x)
    scores = out if not isinstance(out, tuple) else out[1]
    return scores.abs().max().item()


scales = [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
mha_vals = [max_logit_at_weight_scale(StandardMHA, s) for s in scales]
qkA_vals = [max_logit_at_weight_scale(QKNormCosine, s, init_scale=10.0) for s in scales]
qkB_vals = [max_logit_at_weight_scale(QKNormLayerNorm, s) for s in scales]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scales, mha_vals, "o-", label="Standard MHA", color="#EF4444", linewidth=2)
ax.plot(scales, qkA_vals, "s-", label="QK-Norm A (cosine, g=10)", color="#10B981", linewidth=2)
ax.plot(scales, qkB_vals, "^-", label="QK-Norm B (LayerNorm)", color="#1E3A8A", linewidth=2)
ax.set_xlabel("Q/K weight magnitude scale (proxy for training drift)")
ax.set_ylabel("max |attention logit|  (log scale)")
ax.set_yscale("log")
ax.set_title("QK-Norm caps logit magnitude regardless of how large Q/K weights grow")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"At 50× weight scale — standard: {mha_vals[-1]:.1f},  QK-Norm A: {qkA_vals[-1]:.2f},  QK-Norm B: {qkB_vals[-1]:.2f}")


## §6 — Dynamic Tanh (DyT)

Skip the statistics entirely. The empirical observation: LayerNorm's input→output mapping looks like a tanh squash. Replace it with `tanh(α x)` directly.

$$\text{DyT}(x) = \gamma \odot \tanh(\alpha \cdot x) + \beta$$

| Symbol | Shape | Description |
|--------|-------|-------------|
| α | scalar | Learnable steepness |
| γ, β | $(d_{\text{model}},)$ | Learnable per-feature scale and shift |

> No reduction, no division — pure element-wise op.


In [ ]:
class DyT(nn.Module):
    """
    Dynamic Tanh (Zhu, Chen, He, LeCun, Liu — CVPR 2025).

    DyT(x) = γ ⊙ tanh(α · x) + β

    Args:
        d_model:    Feature dimension.
        init_alpha: Initial value of α (paper recommends ~0.5 for transformers).
    """

    def __init__(self, d_model: int, init_alpha: float = 0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(init_alpha))
        self.gamma = nn.Parameter(torch.ones(d_model))   # [d_model]
        self.beta  = nn.Parameter(torch.zeros(d_model))  # [d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.gamma * torch.tanh(self.alpha * x) + self.beta


In [ ]:
# ── visualize the LayerNorm input→output mapping that motivated DyT ─────────
# For a single token, plot input x_k vs normalized output y_k. The mapping is roughly
# a tanh-shaped squash — the "S" that DyT replaces with an actual tanh.

torch.manual_seed(7)
x_tok = torch.randn(d_model) * 3                  # one token's pre-norm vector
ln_single = LayerNorm(d_model)
dyt_single = DyT(d_model, init_alpha=0.5)

with torch.no_grad():
    y_ln  = ln_single(x_tok.view(1, 1, d_model)).view(d_model)
    y_dyt = dyt_single(x_tok.view(1, 1, d_model)).view(d_model)

# Sort by input value for a clean curve
order = x_tok.argsort()
x_sorted   = x_tok[order].numpy()
y_ln_sorted  = y_ln[order].numpy()
y_dyt_sorted = y_dyt[order].numpy()

# Reference: pure tanh(α x)
alpha = dyt_single.alpha.item()
ref_x = np.linspace(x_sorted.min(), x_sorted.max(), 200)
ref_y = np.tanh(alpha * ref_x)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x_sorted, y_ln_sorted,  s=30, alpha=0.7, label="LayerNorm output", color="#1E3A8A")
ax.scatter(x_sorted, y_dyt_sorted, s=30, alpha=0.7, label=f"DyT output (α={alpha:.2f}, γ=1, β=0)", color="#10B981")
ax.plot(ref_x, ref_y, "--", color="#94A3B8", label=f"tanh(α x)", linewidth=1.5)
ax.set_xlabel("Input feature value  x_k")
ax.set_ylabel("Output feature value  y_k")
ax.set_title("Per-feature input→output mapping for a single token")
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color="#cbd5e1", linewidth=0.5)
ax.axvline(0, color="#cbd5e1", linewidth=0.5)
plt.tight_layout()
plt.show()

print("LayerNorm's mapping traces an S-curve through the points;")
print("DyT skips computing μ, σ and uses tanh(α · x) directly.")


## Summary

| Scheme | Axis pooled | Re-center | Re-scale | Params per layer | Modern usage |
|--------|-------------|-----------|----------|------------------|--------------|
| BatchNorm | per-channel over `(N, T)` | yes | yes | `2 · d_model` | CNNs, never LLMs |
| LayerNorm | feature dim, per token | yes | yes | `2 · d_model` | BERT, GPT-2, encoders |
| RMSNorm | feature dim, per token | no | yes | `d_model` | LLaMA, Mistral, Qwen, SmolLM3 |
| Pre-Norm placement | — | — | — | same as norm | every modern decoder LLM |
| Peri-LN placement | — | — | — | `2×` norm params | Gemma 2/3, OLMo 2 |
| QK-Norm A (cosine) | head dim of Q, K | no | yes (unit) | `n_heads` (g) | Henry et al. 2020 |
| QK-Norm B (LayerNorm) | head dim of Q, K | yes | yes | `2 · d_k` per Q/K | ViT-22B, Gemma 2 |
| DyT | none — element-wise | — | learnable tanh | `2 · d_model + 1` | research, CVPR 2025 |

The trajectory: **drop the batch dependency** (LN), **drop the mean** (RMSNorm), **add a second norm post-sublayer** (Peri-LN), **target Q/K only** (QK-Norm), **drop the statistics entirely** (DyT). Each step either removed an operation or shifted where the work happens.
